In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
BASE_PATH = "/Volumes/main/lakehouse_marketing"
RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"

* Definindo Schema e lendo o csv da Raw

In [0]:
conversions_schema = StructType([
        StructField("campaign_id", StringType(), True),
        StructField("conversion_date", StringType(), True),
        StructField("conversion_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("revenue", StringType(), True)
])

df_conversions_raw = spark.read\
                        .option("header", "true")\
                        .csv(f"{RAW_PATH}/conversions")

df_conversions_raw.display()

In [0]:
df_conversions_bronze = df_conversions_raw\
                        .withColumn("ingestion_timestamp", F.current_timestamp())\
                        .withColumn("source_file", F.col("_metadata.file_path"))

* Escrita na Bronze

In [0]:
BRONZE_CONVERSIONS_PATH = f"{BRONZE_PATH}/conversions"

df_conversions_bronze.write\
    .format('delta')\
    .mode('overwrite')\
    .save(BRONZE_CONVERSIONS_PATH)

In [0]:
dbutils.fs.ls(BRONZE_CONVERSIONS_PATH)

In [0]:
display(spark.read\
    .format("delta")\
    .load(f"{BRONZE_CONVERSIONS_PATH}"))